In [1]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

##############################################################################
# 1. Funciones auxiliares: prefix sums, entropía
##############################################################################
def build_prefix_counts(y_sorted):
    """
    Construye una matriz prefix_count de dimensión (n+1, k),
    donde k es el número de clases únicas en y_sorted.
    prefix_count[i, c] = número de ocurrencias de la clase c en y_sorted[:i].
    Retorna:
      - prefix_count (np.ndarray)
      - class2idx (dict) para mapear cada clase a su índice en la matriz
    """
    unique_classes = np.unique(y_sorted)
    class2idx = {cls: i for i, cls in enumerate(unique_classes)}
    k = len(unique_classes)
    n = len(y_sorted)
    
    prefix_count = np.zeros((n+1, k), dtype=np.int64)
    
    for i, val in enumerate(y_sorted, start=1):
        prefix_count[i] = prefix_count[i-1]  # copia la fila anterior
        prefix_count[i, class2idx[val]] += 1
    
    return prefix_count, class2idx

def entropy_from_counts(counts):
    """
    Dada una lista/array 'counts' con el conteo de cada clase,
    calcula la entropía -sum(p_i log p_i).
    Evita log(0) ajustando con un eps.
    """
    total = counts.sum()
    if total == 0:
        return 0.0
    p = counts / total
    # Evitar log(0)
    p[p <= 1e-15] = 1.0
    return -np.sum(p * np.log(p))

def get_partition_entropy(prefix_count, i):
    """
    Dado un índice de corte i (entre 0 y n), calcula la entropía de la partición "izquierda" [0..i) y
    "derecha" [i..n) usando prefix_count. Retorna la entropía combinada, es decir:
                    (nL / n) * ent(L) + (nR / n) * ent(R).
    """
    n = prefix_count.shape[0] - 1  # total de elementos
    # Conteo de cada clase en la part. izq: prefix_count[i]
    left_counts = prefix_count[i]
    nL = i
    
    # Conteo de cada clase en la part. der: prefix_count[n] - prefix_count[i]
    right_counts = prefix_count[n] - prefix_count[i]
    nR = n - i
    
    if nL == 0 or nR == 0:
        # Corte trivial, entropía = entera en una partición
        # Si i==0 => todo a la derecha, si i==n => todo a la izquierda
        return entropy_from_counts(prefix_count[n])  # sin particionar
    eL = entropy_from_counts(left_counts)
    eR = entropy_from_counts(right_counts)
    return (nL / n) * eL + (nR / n) * eR


##############################################################################
# 2. Funciones core MDLP con heurísticas
##############################################################################
def find_best_cut(x_sorted, y_sorted, prefix_count, candidate_indices, min_partition_size=2):
    """
    Busca el mejor índice de corte entre 'candidate_indices'.
    Emplea prefix sums para calcular rápidamente la entropía combinada.
    Retorna (best_i, best_entropy) o (None, None) si no hay corte válido.
    - min_partition_size: evita cortes que dejen particiones demasiado pequeñas.
    """
    n = len(x_sorted)
    best_entropy = np.inf
    best_idx = None
    
    for i in candidate_indices:
        # Evitar particiones muy pequeñas
        if i < min_partition_size or (n - i) < min_partition_size:
            continue
        val_entropy = get_partition_entropy(prefix_count, i)
        if val_entropy < best_entropy:
            best_entropy = val_entropy
            best_idx = i
    return best_idx, best_entropy

def mdl_stop(prefix_count, cut_idx, entropy_after_cut):
    """
    Implementación simplificada de la condición MDL con prefix sums.
    Retorna la ganancia de información si el corte pasa la prueba, de lo contrario, None.
    """
    n = prefix_count.shape[0] - 1
    # entropía sin corte
    total_counts = prefix_count[n]
    es = entropy_from_counts(total_counts)
    gain = es - entropy_after_cut
    
    # para la penalización MDL:
    # conteo de clases totales
    k = np.count_nonzero(total_counts)
    
    # Conteos izq, der
    left_counts = prefix_count[cut_idx]
    k1 = np.count_nonzero(left_counts)
    right_counts = total_counts - left_counts
    k2 = np.count_nonzero(right_counts)
    
    # entropías parciales
    e_left = entropy_from_counts(left_counts)
    e_right = entropy_from_counts(right_counts)
    
    # delta = log(3^k - 2) - (k*es - k1*e_left - k2*e_right)
    delta = np.log(float((3**k) - 2)) - (k*es - k1*e_left - k2*e_right)
    cond = (np.log(float(n - 1)) / n) + (delta / n)
    
    if gain < cond:
        return None
    return gain

def cut_points_recursive(x_sorted, y_sorted, prefix_count, 
                        min_size=2, 
                        max_depth=9999,
                        depth=1,
                        candidate_sample_rate=0.1):
    """
    Encuentra cortes usando recursión y MDL.
      - min_size: tamaño mínimo de partición para seguir cortando.
      - max_depth: profundidad máxima de recursión (poda temprana).
      - candidate_sample_rate: fracción de puntos únicos a considerar como candidatos.
    """
    n = len(x_sorted)
    # Condiciones de parada
    if n < 2 or depth > max_depth:
        return []
    
    # Generamos índices candidatos (heurística):
    # Tomamos índices en [1..(n-1)] donde x cambia, y submuestreamos.
    diffs = np.where(x_sorted[:-1] != x_sorted[1:])[0] + 1
    if len(diffs) == 0:
        return []
    
    # Submuestrear los índices posibles
    sample_size = max(1, int(len(diffs) * candidate_sample_rate))
    if sample_size < len(diffs):
        candidate_indices = np.random.choice(diffs, size=sample_size, replace=False)
    else:
        candidate_indices = diffs
    
    # Buscar el mejor corte
    cut_idx, best_entropy = find_best_cut(x_sorted, y_sorted, prefix_count,
                                          candidate_indices, min_size)
    if cut_idx is None:
        return []
    
    # Chequear MDL
    gain = mdl_stop(prefix_count, cut_idx, best_entropy)
    if gain is None:
        return []
    
    # Aceptamos el corte y continuamos recursivamente
    left_x = x_sorted[:cut_idx]
    left_y = y_sorted[:cut_idx]
    right_x = x_sorted[cut_idx:]
    right_y = y_sorted[cut_idx:]
    
    # prefix counts para cada lado
    prefix_left, _ = build_prefix_counts(left_y)
    prefix_right, _ = build_prefix_counts(right_y)
    
    # Cortes en la parte izquierda (aumentamos profundidad)
    left_cuts = cut_points_recursive(left_x, left_y, prefix_left, 
                                     min_size=min_size, 
                                     max_depth=max_depth,
                                     depth=depth+1,
                                     candidate_sample_rate=candidate_sample_rate)
    
    # Cortes en la parte derecha
    right_cuts = cut_points_recursive(right_x, right_y, prefix_right, 
                                      min_size=min_size, 
                                      max_depth=max_depth,
                                      depth=depth+1,
                                      candidate_sample_rate=candidate_sample_rate)
    
    # El valor real de corte se define como (x_sorted[cut_idx-1] + x_sorted[cut_idx]) / 2, para distinguirlo del índice.
    this_cut_value = 0.5 * (x_sorted[cut_idx - 1] + x_sorted[cut_idx])
    
    # Unimos los cortes de la parte izquierda, el corte actual, y los cortes de la parte derecha
    return left_cuts + [this_cut_value] + right_cuts

def cut_points_mdlp(x, y, min_size=2, max_depth=9999, candidate_sample_rate=0.1):
    """
    Función principal para obtener TODOS los cortes MDLP de una sola variable x.
    Sin límite total de splits; la única poda son: min_size y max_depth.
    """
    # Ordenar x, y simultáneamente
    order_idx = np.argsort(x)
    x_sorted = x[order_idx]
    y_sorted = y[order_idx]
    
    # Construir prefix_count
    prefix_count, _ = build_prefix_counts(y_sorted)
    
    # Llamar recursivo
    cut_vals = cut_points_recursive(x_sorted, y_sorted, prefix_count,
                                    min_size=min_size,
                                    max_depth=max_depth,
                                    candidate_sample_rate=candidate_sample_rate)
    
    if len(cut_vals) == 0:
        return np.array([])
    return np.unique(cut_vals)


##############################################################################
# 3. Paralelización sobre columnas
##############################################################################
def mdlp_parallel(data, target_col, 
                  min_size=2, max_depth=10, candidate_sample_rate=0.1,
                  n_jobs=-1):
    """
    Discretiza cada columna numérica de 'data' (salvo 'target_col') con MDLP.
    Devuelve:
      - dict con {col_name: array de puntos de corte o 'All'}
      - DataFrame discretizado

    Parámetros:
      - min_size: tamaño mínimo de partición
      - max_depth: profundidad máxima de recursión
      - candidate_sample_rate: fracción de puntos únicos a considerar
      - n_jobs: núcleos a utilizar en paralelo (joblib)
    """
    predictor_cols = [c for c in data.columns if c != target_col]
    
    def process_col(col):
        x = data[col].values
        y = data[target_col].values
        
        # Verificar si x es numérico
        if not np.issubdtype(x.dtype, np.number):
            # No discretizamos columnas no numéricas
            return data[col], "non-numeric"
        
        # Obtener cortes
        cuts = cut_points_mdlp(x, y, 
                               min_size=min_size, 
                               max_depth=max_depth, 
                               candidate_sample_rate=candidate_sample_rate)
        if len(cuts) == 0:
            # Sin cortes => todo en un solo bin
            return pd.Series(["All"] * len(x)), "All"
        
        # Construir bins finales (incluyendo min y max)
        bins = np.concatenate(([x.min()], cuts, [x.max()]))
        bins = np.unique(bins)  # Evitar duplicados por si min == max == corte
        if len(bins) == 1:
            return pd.Series(["All"] * len(x)), "All"
        
        # Discretizar usando pd.cut
        disc = pd.cut(x, bins=bins, include_lowest=True, labels=False)
        disc = disc.astype(int) + 1  # para que los bins empiecen en 1
        return disc, cuts
    
    # Paralelizar con joblib
    from joblib import Parallel, delayed
    results = Parallel(n_jobs=n_jobs, batch_size=1, backend="loky")(
        delayed(process_col)(col) for col in predictor_cols
    )
    
    # Armar el DataFrame y diccionario de cortes
    disc_data = data.copy()
    cut_dict = {}
    for col, (col_disc, cuts) in zip(predictor_cols, results):
        disc_data[col] = col_disc
        cut_dict[col] = cuts
    
    return {"cutp": cut_dict, "Disc.data": disc_data}


##############################################################################
# 4. Ejemplo de uso
##############################################################################
if __name__ == "__main__":
    import time
    np.random.seed(42)
    # Cargar dataset
    df = pd.read_csv("C:\\Users\\Carlo\\Desktop\\mdlpChido\\base de datos\\poker.csv")

    start = time.time()
    # Llamada al proceso de discretización sin 'max_splits'
    result = mdlp_parallel(df, 'Class',
                           min_size=40,       # evita particiones muy pequeñas
                           max_depth=20,      # limita profundidad
                           candidate_sample_rate=0.9,  # submuestreo: 80% de puntos
                           n_jobs=-1)
    
    

    print("Cortes encontrados:")
    for col, val in result["cutp"].items():
        print(f" {col}: {val}")
        
    end = time.time()
    print(f"Tiempo total (s): {end - start:.2f}")
    
    # Guardar el DataFrame discretizado en un nuevo archivo CSV
    discretized_file = r"C:\\Users\\Carlo\\Desktop\\mdlpChido\\base de datos con mi MDLP\\poker.csv"
    result["Disc.data"].to_csv(discretized_file, index=False)

    print(f"\nDataFrame discretizado guardado en: {discretized_file}")
    print("Primeras filas del DataFrame discretizado:")
    print(result["Disc.data"].head(10))

Cortes encontrados:
 S1: All
 C1: [ 2.5 11.5]
 S2: All
 C2: [ 3.5 11.5]
 S3: All
 C3: [ 3.5 10.5]
 S4: All
 C4: [ 3.5 11.5]
 S5: All
 C5: [ 3.5 12.5]
Tiempo total (s): 4.41

DataFrame discretizado guardado en: C:\\Users\\Carlo\\Desktop\\mdlpChido\\base de datos con mi MDLP\\poker.csv
Primeras filas del DataFrame discretizado:
    S1  C1   S2  C2   S3  C3   S4  C4   S5  C5  Class
0  All   1  All   3  All   2  All   1  All   2      0
1  All   3  All   1  All   3  All   2  All   2      1
2  All   2  All   2  All   2  All   1  All   2      1
3  All   2  All   3  All   3  All   1  All   2      1
4  All   2  All   2  All   1  All   2  All   2      0
5  All   2  All   2  All   2  All   3  All   2      0
6  All   2  All   2  All   1  All   2  All   2      0
7  All   1  All   2  All   2  All   1  All   2      0
8  All   2  All   3  All   2  All   2  All   2      0
9  All   2  All   2  All   2  All   2  All   2      0
